In [60]:
import ollama

# select from available ollama models
available_models = ollama.list()

# Get only the name from the REST API response
available_models=[model.model for model in available_models.models]
print("Available models: ", available_models)

Available models:  ['nomic-embed-text:latest', 'llama3.2:latest', 'minimax-m3:cloud']


In [61]:
#to-do change template to match Llama3.1 chat response template
template =  { "question": " ", "answer": " "}



In [62]:
import os
import sys
 
import json
from typing import List
from tqdm import tqdm
import PyPDF2
from prompt_toolkit.shortcuts import radiolist_dialog
import re
import ollama
import subprocess
 

In [63]:
from ollama import Client

client = Client(host='http://localhost:11434')
response = client.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)


In [64]:
def fix_json (crptd_json):
    messages = [
    {'role': 'system', 'content': f'You are an API that converts the wrongly formatted JSON into a properly fomatted one by following this template : {template} . Only respond with the JSON and no additional text. \n.'},
    {'role': 'user', 'content': 'Wrong JSON: ' + crptd_json}
    ]

    response = client.chat(
        model="llama3.2:latest", #"gemma:2b" ,
        messages=messages,
        

    )

    response_text = response.message.content.strip()
    # correct corrupted json with rule-based function
    response_text = correct_corrupted_json(response_text)
    try:
        json_data = json.loads(response_text)
        print(json_data)
        return json_data
    except json.JSONDecodeError:
        print("The JSON is not valid, reformatting again.")
        fix_json (crptd_json)
        return []


def correct_corrupted_json(corrupted_json_str):
    # Attempt to correct common JSON issues
    try:
        # Try to parse the JSON directly
        parsed_json = json.loads(corrupted_json_str)
        return parsed_json
    except json.JSONDecodeError:
        # Handle common issues
        
        # Remove unnecessary slashes
        corrected_json_str = corrupted_json_str.replace('\/', '/').replace('\\', '')
        
        # Remove extra commas
        corrected_json_str = corrected_json_str.replace(',}', '}').replace(',]', ']')
        corrected_json_str = corrected_json_str.replace('{,', '{').replace(',]', ']')
        
        # Handle missing closing braces
        if corrected_json_str.strip().endswith('}'):
            pass
        else:
            corrected_json_str += '}'
        
        # Attempt to parse the corrected JSON
        try:
            parsed_json = json.loads(corrected_json_str)
            return parsed_json
        except json.JSONDecodeError as e:
            raise ValueError(f"Unable to correct JSON: {e}")

def is_valid_json_structure(response_text: str) -> bool:
            """Check if the text has valid JSON structure using regex pattern matching."""
            return bool(re.match(r'^[\s]*\{.*\}[\s]*$', response_text, re.DOTALL))


<>:37: SyntaxWarning: invalid escape sequence '\/'
<>:37: SyntaxWarning: invalid escape sequence '\/'
/var/folders/m0/plcrj91s5rj8_vj1vchj_cq40000gn/T/ipykernel_16442/1996729778.py:37: SyntaxWarning: invalid escape sequence '\/'
  corrected_json_str = corrupted_json_str.replace('\/', '/').replace('\\', '')


In [65]:
failed_responses = []  # List to store failed responses

In [66]:
def generate_questions_answers(text_chunk, model="llama3.2:latest"):

    messages = [
    {'role': 'system', 'content': 'You are an API that converts bodies of text into a single question and answer into a valid JSON format. Each JSON " \
    "should contain a single question with a single answer. Only respond with valid JSON and no additional text. Test the JSON for validity before returning it. I will be very disaoppinted with any errors in the JSON.  \n.'},
    {'role': 'user', 'content': 'Text: ' + text_chunk}
    ]

    response = client.chat(
        model=model,#"gemma:2b",
        messages=messages,
        
    )

    response_text = response.message.content.strip()

    # add check here to see if response_text is valid JSON before calling json.loads and skip if invalid
    if not is_valid_json_structure(response_text):
        print("Invalid JSON structure")
        print("Failed to process",len(response_text))
        failed_responses.append(response_text)  # Store the failed response
        
        failed_responses.append("\n") 
        return []


    try:
        #test response_text for json validity before calling json.loads and skip if invalid
       
        # Check if response_text has valid JSON structure
        if not is_valid_json_structure(response_text):
            print("Invalid JSON structure")
            print("Failed to process",len(response_text))
            failed_responses.append(response_text)  # Store the failed response
            failed_responses.append("\n")    
    
    
            return []
        if not response_text.strip().startswith('{') or not response_text.strip().endswith('}'):
            print("Invalid JSON format - missing braces")
            print("Failed to process",len(response_text))
            failed_responses.append(response_text)  # Store the failed response
            failed_responses.append("\n") 
            return []
        
        json_data = json.loads(response_text)
        print(json_data)
        return json_data
    except json.JSONDecodeError as e:
        print(str(e))
        print("Error: Response is not valid JSON.... Trying to fix the JSON.")
        print("Failed to process",len(response_text))
        print(response_text)
        failed_responses.append(response_text)  # Store the failed response
        failed_responses.append("\n") 
        #correct_corrupted_json(response_text)
        #fix_json(response_text)
        return []
    finally:
        #continue and skip this value
        pass

In [67]:
all_responses = []

In [68]:
from typing import List, Dict, Any
import fitz  # PyMuPDF
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

def extract_text_from_pdf(file_path):
    pdf_file_obj = open(file_path, 'rb')
    pdf_reader = PyPDF2.PdfReader(pdf_file_obj)
    text = ''
    for page_num in range(len(pdf_reader.pages)):
        page_obj = pdf_reader.pages[page_num]
        text += page_obj.extract_text()
    pdf_file_obj.close()
    #print first few characters of the text
    print(text[:1000])
    return text

def process_text(text: str, chunk_size: int = 250) -> List[dict]:
    text_chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    # Clean and prepare text
    #paragraphs = [p.strip() for p in text.split('\n') if len(p.strip()) > 50]
    #print(f"\nFound {len(paragraphs)} Customer agreement paragraphs")
    #print(f"Average paragraph length: {sum(len(p) for p in paragraphs) // len(paragraphs)} characters")
    #print(f"\nSample paragraph:")
    #print(paragraphs[0][:200] + "...")

    
    for chunk in tqdm(text_chunks, desc="Processing chunks", unit="chunk"):
        response = generate_questions_answers(chunk)
        if 'question' in response and 'answer' in response:
            all_responses.append({'question': response['question'], 'answer': response['answer']})
             
    print(f"Processed {len(all_responses)} responses.")
    #print first few responses
    print(all_responses[:3])
    return all_responses



In [69]:
import json
import csv
import os
import sys
import argparse
import shutil

def convert_json_to_csv(source_documents, output_datasets):
  
    # Directory of PDFs
    json_files = [os.path.splitext(f)[0] for f in os.listdir(source_documents) 
            if f.endswith('.json')]
    if not json_files:
        print(f"No JSON files found in directory '{source_documents}'.")
        sys.exit(1) 

    instruction = "You are a helpful assistant who can answer questions about the topic in the dataset."

    # Process each PDF's JSON file
    for json_file in json_files:
        # Construct full paths for input and output files
        json_file = os.path.join(source_documents, json_file + '.json')
        csv_file = os.path.join(output_datasets, os.path.splitext(os.path.basename(json_file))[0] + '.csv')
        print("Processing:", json_file)
        print("Output:", csv_file)
        try:
            # Read the JSON file
            with open(json_file, 'r', encoding='utf-8') as f:
                responses = json.load(f)

            # Write to CSV file
            with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
                fieldnames = ['prompt', 'question', 'answer']
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                for response in responses['responses']:
                    if 'question' in response and 'answer' in response:
                        #to-do write in model template
                        writer.writerow({
                            'prompt': instruction,
                            'question': response['question'],
                            'answer': response['answer']
                        })
            print(f"Created CSV file: {csv_file}")
            # Move JSON files to output directory if CSV was created successfully
            if os.path.exists(csv_file):
                output_json = os.path.join(output_datasets, os.path.basename(json_file))
                try:
                    shutil.move(json_file, output_json)
                    print(f"Moved JSON file to: {output_json}")
                except Exception as e:
                    print(f"Error moving JSON file {json_file}: {str(e)}")
        except Exception as e:
            print(f"Error processing {json_file}: {str(e)}")
    print("All JSONs processed successfully.")

In [70]:
pdfs = []
arg_type = "--dir"
sourcepath = "../source"

In [71]:
 
def get_pdfs(sourcepath="../source"):
    if arg_type == '--file':
        # Process single PDF
        if not os.path.exists(path):
            print(f"Error: PDF file '{sourcepath}' not found.")
            sys.exit(1)
        pdfs = [sourcepath]
    elif arg_type == '--dir':
        # Process all PDFs in specified directory
        if not os.path.exists(sourcepath):
            print(f"Error: Directory '{sourcepath}' not found.")
            sys.exit(1)
        pdfs = [os.path.join(sourcepath, f) for f in os.listdir(sourcepath) if f.endswith('.pdf')]
        if not pdfs:
            print(f"No PDF files found in directory '{sourcepath}'.")
            sys.exit(1)
    else:
        print("Invalid argument. Use --file or --dir.")
        sys.exit(1)
    return pdfs


In [72]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

In [73]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# This step converts cleaned page-level text into paragraph-level records.

def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [74]:
# main function is here


# select from available ollama models
available_models = ollama.list()

# Get only the name from the REST API response
available_models=[model.model for model in available_models.models]
print("Available models: ", available_models)
    
# Prompt the user to select the model for Q&A generation
model = "llama3.2:latest"

# get pdfs from the commandline arguments
pdfs = get_pdfs()
# Process each PDF
for pdf in pdfs:
    print(f"Processing PDF: {pdf}")
    try:
        text = extract_text_from_pdf(pdf)

        pdf_pages = extract_pdf_pages("../source/deposit-account-agreement.pdf")
        print(f"Total pages with extracted text: {len(pdf_pages)}")
        print("Page-level character counts:")
        for item in pdf_pages:
            print(f"Page {item['page']}: {item['char_count']} characters")

        cleaned_pages = []
        for page in pdf_pages:
            cleaned_text = clean_pdf_text(page["text"])
            cleaned_pages.append({
                "page": page["page"],
                "text": cleaned_text,
                "char_count": len(cleaned_text),
        })
        print("Total cleaned pages:", len(cleaned_pages))

        paragraph_records = split_into_paragraph_records(cleaned_pages)
        print("Total paragraph records:", len(paragraph_records))
        pdf = os.path.splitext(pdf)[0]
        count = 0
        for record in paragraph_records[:8]:
            print("=" * 80)
            print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
            print(record["text"])
            responses = process_text(record["text"]) #{"responses": process_text(record["text"]), "model": model}
          
            count = count + len(responses)
            print(f"Total responses processed  {count}")

    except FileNotFoundError:
        print(f"Error: PDF file '{pdf}' not found.")
        continue
    except PyPDF2.PdfReadError:
        print(f"Error: Unable to read PDF file '{pdf}'. File may be corrupted or invalid.")
        continue
    except Exception as e:
        print(f"Unexpected error while reading PDF: {str(e)}")
        continue

    responses = {"responses": all_responses}
    # Save responses to JSON file
    # strip the .pdf extension from the pdf file name
    pdf = os.path.splitext(pdf)[0]
    with open(f'{pdf}.json', 'w') as f:
        json.dump(responses, f, indent=2)
        print(f"Saved responses to: {pdf}.json")

    with open(f'{pdf}_failedresponses.jsonl', 'w') as f:
            json.dump(failed_responses, f, indent=2)
            print(f"Saved responses to: {pdf}_failedresponses.json")

    
print("All PDFs processed successfully.")

Available models:  ['nomic-embed-text:latest', 'llama3.2:latest', 'minimax-m3:cloud']
Processing PDF: ../source/deposit-account-agreement.pdf
DEPOSIT ACCOUNT AGREEMENT
JPMorgan Chase Bank, N.A. Member FDIC© 2026 JPMorgan Chase & Co.Page 1 of 30
Effective 6/14/2026DEPOSIT ACCOUNT AGREEMENT  
AND PRIVACY NOTICE
Thank you for choosing Chase
This is your Deposit Account Agreement, or contract, with us. This agreement applies to all Chase personal and business and J.P. Morgan Private Client 
deposit accounts and the terms and conditions are identical for all of these accounts.
We recommend keeping this agreement but we regularly update it, so you can always get the current agreement at chase.com , a branch or by request 
when you call us.The Deposit Account Agreement also includes these separate documents that pertain to our personal and business accounts:
• Rates for interest-bearing accounts
• Personal accounts:
 -Additional Banking Services and Fees for Personal Accounts
 -J.P. Morgan Pr

Processing chunks:  11%|█         | 1/9 [00:00<00:06,  1.22chunk/s]

{'question': 'What is the effective date of my deposit account agreement?', 'answer': 'Effective June 14, 2026'}


Processing chunks:  22%|██▏       | 2/9 [00:02<00:08,  1.16s/chunk]

{'question': 'What is the effect on my Chase or J.P. Morgan Private Client account if the terms and conditions of this agreement are updated?', 'answer': 'The terms and conditions may be changed, and you should review and agree to any new terms.'}


...

{'question': 'What happens to a joint account with rights of survivorship?', 'answer': 'The account continues as a joint account with rights of survivorship among the remaining owners.'}
Processed 123 responses.
[{'question': 'What is the effective date of my deposit account agreement?', 'answer': 'Effective June 14, 2026'}, {'question': 'What is the effect on my Chase or J.P. Morgan Private Client account if the terms and conditions of this agreement are updated?', 'answer': 'The terms and conditions may be changed, and you should review and agree to any new terms.'}, {'question': 'What information can I find on the Deposit Account Agreement website?', 'answer': 'The Deposit Account Agreement also includes these separate documents that pertain to our personal and business accounts:'}]
Total responses processed  575
Page: 6 | Paragraph: 5 | Characters: 500
Joint account with no right of survivorship (also called “tenants in common”) If a joint account does not have rights of survivors

Processing chunks:  50%|█████     | 1/2 [00:01<00:01,  1.02s/chunk]

{'question': 'What happens to an interest in a joint account without rights of survivorship when one joint owner dies?', 'answer': "The deceased owner's interest passes to the owner's estate."}


Processing chunks: 100%|██████████| 2/2 [00:02<00:00,  1.02s/chunk]

{'question': "What are the implications of designating an account as 'Tenants in common' or 'JTIC'?", 'answer': 'This type of account does not have rights of survivorship.'}
Processed 125 responses.
[{'question': 'What is the effective date of my deposit account agreement?', 'answer': 'Effective June 14, 2026'}, {'question': 'What is the effect on my Chase or J.P. Morgan Private Client account if the terms and conditions of this agreement are updated?', 'answer': 'The terms and conditions may be changed, and you should review and agree to any new terms.'}, {'question': 'What information can I find on the Deposit Account Agreement website?', 'answer': 'The Deposit Account Agreement also includes these separate documents that pertain to our personal and business accounts:'}]
Total responses processed  700
Saved responses to: ../source/deposit-account-agreement.json
Saved responses to: ../source/deposit-account-agreement_failedresponses.json
All PDFs processed successfully.


In [75]:
#!uv pip install unsloth
#!uv pip install transformers trl datasets peft bitsandbytes accelerate torch sentencepiece protobuf

In [76]:
convert_json_to_csv("../source/", "../data/")
import pandas as pd

df = pd.read_csv('../data/deposit-account-agreement.csv')
df.head()
df.count()
# Write to a .jsonl file
with open("../data/deposit-account-agreement.jsonl", "w", encoding="utf-8") as f:
    for record in df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

Processing: ../source/deposit-account-agreement.json
Output: ../data/deposit-account-agreement.csv
Created CSV file: ../data/deposit-account-agreement.csv
Moved JSON file to: ../data/deposit-account-agreement.json
All JSONs processed successfully.


In [77]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/Users/nkoneru/vscode/WorkingSamples/Project-one/Project-one/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


/Users/nkoneru/vscode/WorkingSamples/Project-one/Project-one/venv/lib/python3.12/site-packages/trl/extras/best_of_n_sampler.py:24: FutureWarning: `BestOfNSampler` is deprecated and will be removed in TRL 0.25.
  class BestOfNSampler:


In [80]:
# Load instruction dataset
instruction_data = []
try:
    with open('../data/deposit-account-agreement.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            instruction_data.append(json.loads(line))
    print(f"Loaded {len(instruction_data)} instruction examples")
except FileNotFoundError:
    print("File not found. Using embedded data...")
    

print(f"Sample instruction:")
print(f"Q: {instruction_data[10]['question']}")
print(f"A: {instruction_data[10]['answer'][:200]}...")

Loaded 125 instruction examples
Sample instruction:
Q: How do I open a new account
A: You can open your account by following the instructions provided during the sign-up process....
